# H20b: Curvature dependence of checkpoint loss-ratio compounding (toy study)

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/H20b_COMPOUNDING_CURVATURE_DEPENDENCE/run_experiment.py`

This notebook is the reporting front-end for the script above. All core computation is delegated to the script's `run_experiment()` function so that the notebook and script remain behaviorally aligned.

## Truthful scope

This second-pass version keeps the current deep-linear and ReLU branches intact while repairing the biggest control flaw from the first pass:

- the old **legacy quadratic** with fake `1x4` layers is retained only as a continuity reference,
- a new **same-shape constant-Hessian control** with two real `4x4` parameter layers is added as the primary control.

The study still measures only:

1. checkpoint losses for Muon and Frobenius-normalized SGD after separate learning-rate sweeps,
2. checkpoint ratios `loss_normsgd / loss_muon`,
3. fitted slopes of `log(ratio)` versus checkpoint.

Ratios greater than `1` favor Muon. Even with the repaired control, these are still **performance summaries under the present protocol**, not direct measurements of curvature causality.


In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

EXPERIMENT_RELATIVE_PATH = Path(
    "experiments/Tier_1_Core_Mechanism_Tests/H20b_COMPOUNDING_CURVATURE_DEPENDENCE/run_experiment.py"
)

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
for root in candidate_roots:
    candidate = root / EXPERIMENT_RELATIVE_PATH
    if candidate.exists():
        REPO_ROOT = root
        SCRIPT_PATH = candidate
        break
else:
    raise FileNotFoundError(
        "Could not locate the H20b script by searching upward from the current working directory. "
        "Start the notebook from somewhere inside the repository or update EXPERIMENT_RELATIVE_PATH."
    )

spec = importlib.util.spec_from_file_location("h20b_compounding_curvature", SCRIPT_PATH)
h20b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h20b)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Repo root:", REPO_ROOT)
print("Script path:", SCRIPT_PATH)
print("Python executable:", sys.executable)
print("Python version:", platform.python_version())


## Reproducibility and runtime configuration

The next cell executes the same core experiment as the script and stores the structured results dictionary returned by `run_experiment()`. The default numerical setup is intentionally preserved from the retained baseline:

- `DIM=4`
- `NUM_STEPS=500`
- `NUM_SEEDS=5`
- `BATCH_SIZE=64`
- `NS_ITERS=5`
- `MOMENTUM=0.9`
- checkpoints `{100, 200, 300, 400, 500}`
- LR selection at step `200`

The new scientific change in this pass is the **additional same-shape constant-Hessian control**, not a broad redesign of the rest of the experiment.


In [ ]:
notebook_start = time.time()
results = h20b.run_experiment(verbose=False)
notebook_runtime = time.time() - notebook_start

config = results["config"]
surface_order = results["surface_order"]
primary_surface_order = results["primary_surface_order"]
surface_metadata = results["surface_metadata"]
analysis_targets = results["analysis_targets"]
legacy_check = results["method_checks"]["legacy_row_vector_muon_equals_frobenius"]
matrix_check = results["method_checks"]["matrix_layer_muon_differs_from_frobenius"]

print("Experiment ID:", results["experiment_id"])
print("Measured quantities:")
for item in results["measured_quantities"]:
    print("  -", item)
print("Not measured directly:")
for item in results["not_measured_directly"]:
    print("  -", item)
print()
print("Configuration:")
print(f"  dim={config['dim']}, steps={config['num_steps']}, seeds={config['num_seeds']}, batch_size={config['batch_size']}")
print(f"  ns_iters={config['ns_iters']}, momentum={config['momentum']}")
print(f"  checkpoints={config['checkpoints']}, lr_selection_step={config['lr_selection_step']}")
print(f"  seeds={config['seeds']}")
print()
print("Analysis targets:")
print(f"  legacy control = {analysis_targets['legacy_control']}")
print(f"  primary control = {analysis_targets['primary_control']}")
print()
print(f"run_experiment runtime: {results['runtime_seconds']:.2f} s")
print(f"Notebook cell wall-clock runtime: {notebook_runtime:.2f} s")
print(
    "Method checks: legacy row-vector diff = "
    f"{legacy_check['frobenius_diff']:.3e}; generic 4x4 matrix diff = {matrix_check['frobenius_diff']:.3e}"
)


## Surface inventory and explicit control diagnostics

The surface table below is intentionally literal about what is currently implemented.

- **LegacyQuadratic** is the old degenerate control with only `8` parameters and fake `1x4` layer blocks.
- **ConstantHessian** is the new primary repaired control with the same `4x4` / `4x4` layer structure and `32` parameters as the network branches.

This does **not** turn the notebook into a causal proof about curvature. It does, however, remove the most obvious control degeneracy: Muon and NormSGD are no longer trivially identical on the primary constant-Hessian control at fixed LR.


In [ ]:
def format_scalar(value, digits=6):
    value = float(value)
    if not np.isfinite(value):
        return "nan"
    return f"{value:.{digits}f}"


def format_ratio(value, digits=3):
    value = float(value)
    if not np.isfinite(value):
        return "nan"
    return f"{value:.{digits}f}x"


def print_table(headers, rows):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, item in enumerate(row):
            widths[i] = max(widths[i], len(str(item)))
    header_line = " | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers))
    sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))
    print(header_line)
    print(sep_line)
    for row in rows:
        print(" | ".join(str(item).ljust(widths[i]) for i, item in enumerate(row)))


inventory_rows = []
for sname in surface_order:
    meta = surface_metadata[sname]
    inventory_rows.append([
        meta["display_name"],
        meta["analysis_role"],
        meta["family"],
        meta["n_params"],
        meta["layer_shapes"],
        meta["scope_note"],
    ])

print("Surface inventory:")
print_table(
    ["surface", "role", "family", "n_params", "layer_shapes", "scope_note"],
    inventory_rows,
)
print()
print("Method checks:")
print(f"  Legacy row-vector identity check: shape={legacy_check['shape']}, frobenius_diff={legacy_check['frobenius_diff']:.3e}")
print(f"  Generic 4x4 matrix non-identity check: shape={matrix_check['shape']}, frobenius_diff={matrix_check['frobenius_diff']:.3e}")
print()

control_diag = results["seed_results"][0]["surface_results"][analysis_targets["primary_control"]]["surface_diagnostics"]
legacy_diag = results["seed_results"][0]["surface_results"][analysis_targets["legacy_control"]]["surface_diagnostics"]
print("Primary constant-Hessian control diagnostics from the first seed:")
layer_rows = []
for idx, stats in enumerate(control_diag["layer_hessian_stats"], start=1):
    layer_rows.append([
        idx,
        format_scalar(stats["min_eig"], digits=4),
        format_scalar(stats["max_eig"], digits=4),
        format_scalar(stats["condition_number"], digits=4),
    ])
print_table(["layer", "min_eig", "max_eig", "condition_number"], layer_rows)
print()
print("Legacy quadratic first-seed Hessian summary:")
print(legacy_diag)


## Checkpoint loss-ratio trajectories

The figure plots the mean checkpoint ratio across seeds for each surface, with one standard deviation error bars. These are **performance summaries under separately tuned learning rates**, not direct curvature measurements.

Values above `1` favor Muon. The legacy and primary controls are shown separately so that the degeneracy repair is visible rather than buried.


In [ ]:
checkpoints = config["checkpoints"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, sname in zip(axes, surface_order):
    summary = results["surface_summaries"][sname]
    means = [summary["mean_ratios"][cp] for cp in checkpoints]
    stds = [summary["std_ratios"][cp] for cp in checkpoints]
    ax.errorbar(checkpoints, means, yerr=stds, marker="o", linewidth=2, capsize=4)
    ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
    ax.set_title(surface_metadata[sname]["display_name"])
    ax.set_xlabel("Step")
    ax.set_ylabel("NormSGD / Muon loss ratio")
    ax.grid(alpha=0.3)

fig.suptitle("Checkpoint loss-ratio trajectories (mean ± 1 std across seeds)", y=0.98)
plt.tight_layout()
plt.show()

ratio_rows = []
for sname in surface_order:
    summary = results["surface_summaries"][sname]
    ratio_rows.append([
        surface_metadata[sname]["display_name"],
        *[format_ratio(summary["mean_ratios"][cp]) for cp in checkpoints],
        format_scalar(summary["slope_from_mean_ratios"], digits=6),
    ])

print("Mean checkpoint ratios and fitted slope from mean ratios:")
print_table(
    ["surface", *[f"T={cp}" for cp in checkpoints], "slope_from_mean_ratios"],
    ratio_rows,
)


## Primary comparison focus: same-shape constant-Hessian control vs nonlinear branches

Because the legacy quadratic is retained only as a reference, the next figure focuses on the primary analysis triplet:

- `ConstantHessian` (repaired same-shape control)
- `DeepLinear`
- `ReLU_MLP`

This is the comparison used for the primary heuristic ordering tests reported later.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
for sname, marker in zip(primary_surface_order, ["o", "s", "^"]):
    summary = results["surface_summaries"][sname]
    means = [summary["mean_ratios"][cp] for cp in checkpoints]
    ax.plot(checkpoints, means, marker=marker, linewidth=2, label=surface_metadata[sname]["display_name"])
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Step")
ax.set_ylabel("Mean NormSGD / Muon loss ratio")
ax.set_title("Primary comparison after control repair")
ax.grid(alpha=0.3)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

primary_rows = []
for sname in primary_surface_order:
    summary = results["surface_summaries"][sname]
    primary_rows.append([
        surface_metadata[sname]["display_name"],
        format_ratio(summary["mean_ratios"][checkpoints[-1]]),
        format_scalar(summary["slope_from_mean_ratios"], digits=6),
        format_scalar(summary["per_seed_slope_mean"], digits=6),
        format_scalar(summary["per_seed_slope_std"], digits=6),
    ])
print("Primary comparison summary:")
print_table(
    ["surface", f"final ratio @ T={checkpoints[-1]}", "slope_from_mean_ratios", "per_seed_slope_mean", "per_seed_slope_std"],
    primary_rows,
)


## Slope summaries

The current compounding summary statistic is the slope of `log(ratio)` versus checkpoint. Two versions are shown:

1. `slope_from_mean_ratios`: fit after averaging the ratios across seeds,
2. `per_seed_slope_mean ± per_seed_slope_std`: fit within each seed and then summarized across seeds.

The seed-wise summary is more transparent about variability. The sign and magnitude of these slopes should still be interpreted cautiously because the experiment tunes learning rates separately for the two optimizers.


In [ ]:
x = np.arange(len(surface_order))
slope_means = [results["surface_summaries"][s]["per_seed_slope_mean"] for s in surface_order]
slope_stds = [results["surface_summaries"][s]["per_seed_slope_std"] for s in surface_order]
mean_ratio_slopes = [results["surface_summaries"][s]["slope_from_mean_ratios"] for s in surface_order]
labels = [surface_metadata[s]["display_name"] for s in surface_order]

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(x, slope_means, yerr=slope_stds, capsize=5, alpha=0.75, label="per-seed slope mean ± std")
ax.scatter(x, mean_ratio_slopes, color="black", marker="D", s=60, label="slope from mean ratios")
ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_ylabel("Slope of log(ratio) vs checkpoint")
ax.set_title("Compounding-rate summaries")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

slope_rows = []
for sname in surface_order:
    summary = results["surface_summaries"][sname]
    slope_rows.append([
        surface_metadata[sname]["display_name"],
        format_scalar(summary["slope_from_mean_ratios"], digits=6),
        format_scalar(summary["per_seed_slope_mean"], digits=6),
        format_scalar(summary["per_seed_slope_std"], digits=6),
    ])
print("Slope summary table:")
print_table(
    ["surface", "slope_from_mean_ratios", "per_seed_slope_mean", "per_seed_slope_std"],
    slope_rows,
)


## Learning-rate summary

The experiment still selects a "best" learning rate on a fixed discrete grid using the loss at step `200`. The table and figure below therefore document the actual selection protocol rather than implying a globally optimal LR.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
axes = axes.ravel()
seed_positions = np.arange(len(config["seeds"]))

for ax, sname in zip(axes, surface_order):
    summary = results["surface_summaries"][sname]
    muon_lrs = summary["best_lrs"]["muon"]
    norm_lrs = summary["best_lrs"]["normsgd"]
    ax.scatter(seed_positions - 0.08, muon_lrs, label="Muon", s=50)
    ax.scatter(seed_positions + 0.08, norm_lrs, label="NormSGD", s=50)
    ax.set_yscale("log")
    ax.set_xticks(seed_positions)
    ax.set_xticklabels([str(s) for s in config["seeds"]], rotation=45)
    ax.set_xlabel("Seed")
    ax.set_title(surface_metadata[sname]["display_name"])
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Selected learning rate")
axes[-1].legend(loc="best")
fig.suptitle("Best LR selected at step 200 for each seed", y=0.98)
plt.tight_layout()
plt.show()

lr_rows = []
for sname in surface_order:
    summary = results["surface_summaries"][sname]
    lr_rows.append([
        surface_metadata[sname]["display_name"],
        format_scalar(summary["best_lr_medians"]["muon"], digits=6),
        format_scalar(summary["best_lr_medians"]["normsgd"], digits=6),
        [format_scalar(v, digits=6) for v in summary["best_lrs"]["muon"]],
        [format_scalar(v, digits=6) for v in summary["best_lrs"]["normsgd"]],
    ])
print("Selected LR summary:")
print_table(
    ["surface", "median best lr (Muon)", "median best lr (NormSGD)", "Muon best lrs by seed", "NormSGD best lrs by seed"],
    lr_rows,
)


## Control caveats: what changed, and what still does not follow

### What changed in this pass

- The old legacy quadratic is no longer the only control.
- The new `ConstantHessian` branch matches the `4x4` / `4x4` parameter layering of `DeepLinear` and `ReLU_MLP`.
- Muon and NormSGD are therefore no longer trivially identical on the primary control's layer gradients.

### What still does **not** follow

- The experiment still does **not** directly measure curvature along the training trajectory.
- A positive ordering against the repaired constant-Hessian control would still be only a **heuristic performance pattern**, not a proof that nonlinearity or curvature is the cause.
- Because learning rates are tuned separately, part of any advantage may reflect the LR-selection protocol rather than the transform alone.


In [ ]:
primary_t1 = results["tests"]["T1_relu_gt_deep_linear_gt_constant_hessian_mean_slope"]
primary_t2 = results["tests"]["T2_relu_gt_2x_constant_hessian_mean_slope"]
legacy_t1 = results["tests"]["Legacy_reference_T1_relu_gt_deep_linear_gt_legacy_quadratic_mean_slope"]
legacy_t2 = results["tests"]["Legacy_reference_T2_relu_gt_2x_legacy_quadratic_mean_slope"]

print("Legacy vs repaired-control comparison:")
print(f"  Legacy T1: {'PASS' if legacy_t1['passed'] else 'FAIL'} -> {legacy_t1['values']}")
print(f"  Legacy T2: {'PASS' if legacy_t2['passed'] else 'FAIL'} -> {legacy_t2['values']}")
print(f"  Primary T1: {'PASS' if primary_t1['passed'] else 'FAIL'} -> {primary_t1['values']}")
print(f"  Primary T2: {'PASS' if primary_t2['passed'] else 'FAIL'} -> {primary_t2['values']}")
print()
print("Interpretive note:")
print(
    "  The control repair materially changes the heuristic ordering outcome: the legacy 1x4 control produced "
    "mixed/negative ordering tests, whereas the same-shape constant-Hessian control yields positive ordering "
    "tests in this run. That is evidence that the old control was seriously misleading, not that curvature "
    "causality has been proven."
)
print(
    "  Absolute slope magnitudes on DeepLinear and ReLU_MLP remain very small, so the result should be read as "
    "a modest protocol-level separation relative to the repaired control, not as a decisive mechanistic effect."
)


## Final conclusion tied to the actual observed outputs

The next cell reports the primary same-shape-control tests and the legacy continuity checks exactly as computed. A mixed conclusion is acceptable here: the purpose of this notebook is honest reporting of what the current toy computation does and does not support.


In [ ]:
print("Primary same-shape-control tests:")
print(f"  T1 (ReLU > DeepLinear > ConstantHessian by mean-ratio slope): {'PASS' if primary_t1['passed'] else 'FAIL'}")
print(f"     values = {primary_t1['values']}")
print(f"  T2 (ReLU slope > 2x ConstantHessian slope): {'PASS' if primary_t2['passed'] else 'FAIL'}")
print(f"     values = {primary_t2['values']}")
print()
print("Legacy continuity checks:")
print(f"  Legacy T1: {'PASS' if legacy_t1['passed'] else 'FAIL'}")
print(f"  Legacy T2: {'PASS' if legacy_t2['passed'] else 'FAIL'}")
print()

relu_summary = results["surface_summaries"]["ReLU_MLP"]
linear_summary = results["surface_summaries"]["DeepLinear"]
control_summary = results["surface_summaries"][analysis_targets["primary_control"]]

print("Conclusion:")
if primary_t1["passed"] and primary_t2["passed"]:
    print(
        "  Relative to the repaired same-shape constant-Hessian control, the current toy protocol now shows "
        "slightly more favorable slope summaries for the nonlinear branches, and the ReLU branch is positive "
        "while the repaired control is negative."
    )
    print(
        "  However, the absolute nonlinear slopes remain tiny "
        f"(DeepLinear={linear_summary['slope_from_mean_ratios']:.6f}, ReLU_MLP={relu_summary['slope_from_mean_ratios']:.6f}), "
        "so this still does not justify a strong claim that compounding requires curvature."
    )
    print(
        "  The scientifically defensible update is narrower: repairing the control materially changes the pair's "
        "heuristic ordering result, which means the old degenerate quadratic was not an adequate baseline."
    )
else:
    print(
        "  Even after the control repair, this toy study does not support a strong curvature-dependence claim. "
        "Treat it as an honest protocol-level comparison rather than a mechanistic demonstration."
    )

print()
print("Practical takeaway:")
print(
    "  The most meaningful result of this second pass is the improved baseline itself: the notebook/script pair now "
    "distinguishes a degenerate legacy control from a same-shape constant-Hessian control and reports conclusions "
    "against the latter without overclaiming causality."
)
